# RAMEC — Revisión Automática Multimodal de Entregables de Construcción

Notebook de **ejecución en Google Colab**: clona el repositorio, instala las dependencias,
descarga los pesos del modelo y ejecuta las validaciones.

**Antes de empezar:**

(Opcional) Activa GPU en *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
   La inferencia también funciona en CPU.

## 1. Configuración

In [93]:
REPO = "BRIDERI/Ramec"
TAG  = "v1.0"              # etiqueta del Release que contiene los pesos best.pt

## 2. (Opcional) Verificar GPU

In [94]:
!nvidia-smi || echo "Sin GPU: se usará CPU (la inferencia funciona igual)."

/bin/bash: line 1: nvidia-smi: command not found
Sin GPU: se usará CPU (la inferencia funciona igual).


## 3. Clonar el repositorio

In [95]:
!git clone https://github.com/{REPO}.git ramec
%cd ramec

Cloning into 'ramec'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 67 (delta 28), reused 34 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 43.91 KiB | 3.38 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/ramec/ramec/ramec/ramec/ramec/ramec


## 4. Dependencias del sistema (Tesseract + Poppler)
No se instalan con pip; el idioma `spa` es necesario para leer texto en español.

In [96]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 5. Dependencias de Python

In [85]:
!pip -q install -r requirements.txt

## 6. Descargar los pesos del modelo (desde el Release)
Deja `models/planos/best.pt` y `models/documentos/best.pt`.

In [97]:
!REPO={REPO} TAG={TAG} bash scripts/download_models.sh

Descargando pesos desde https://github.com/BRIDERI/Ramec/releases/download/v1.0 ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.8M  100 38.8M    0     0  49.0M      0 --:--:-- --:--:-- --:--:-- 95.5M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.7M  100 38.7M    0     0  51.4M      0 --:--:-- --:--:-- --:--:--  188M
Listo. Pesos descargados:
-rw-r--r-- 1 root root 39M Jul  6 22:07 models/documentos/best.pt
-rw-r--r-- 1 root root 39M Jul  6 22:07 models/planos/best.pt


## 7. Subir los PDFs a revisar
Sube uno o más PDFs (planos y/o documentos). Se guardan en la carpeta `pdfs/`.

In [101]:
import os
os.makedirs("pdfs", exist_ok=True)
from google.colab import files
print("Selecciona uno o más PDFs...")
subidos = files.upload()
for nombre in subidos:
    os.replace(nombre, os.path.join("pdfs", nombre))
!ls -lh pdfs

Selecciona uno o más PDFs...


Saving AVP-SAV-3000-D-EST-GEO-ING-000003.pdf to AVP-SAV-3000-D-EST-GEO-ING-000003.pdf
total 139M
-rw-r--r-- 1 root root 60M Jul  6 22:25 AVP-SAV-3000-D-EST-GEO-ING-000003.pdf
-rw-r--r-- 1 root root 79M Jul  6 22:14 AVP-SAV-9000-D-INF-CYT-ING-000001.pdf


*Alternativa:* en vez de subir archivos, puedes montar tu Google Drive y apuntar a una
carpeta con PDFs (descomenta y ajusta la ruta).

In [58]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p pdfs && cp /content/drive/MyDrive/ruta/a/tus/pdfs/*.pdf pdfs/

## 8. Ejecutar la validación

In [102]:
!python src/infer.py --pdfs pdfs --salida outputs/Reporte_validacion.xlsx

Listo: outputs/Reporte_validacion.xlsx
  planos=0 documentos=2 validacion_profesional=2 filas estandar=2


## 9. Ver y descargar el reporte
Se genera un Excel con seis hojas de verificación.

In [103]:
import pandas as pd
xls = pd.ExcelFile("outputs/Reporte_validacion.xlsx")
print("Hojas:", xls.sheet_names)
display(pd.read_excel(xls, "VALIDACION_PROFESIONAL"))

from google.colab import files
files.download("outputs/Reporte_validacion.xlsx")

Hojas: ['ESTANDAR NOMENCLATURA', 'COMPATIBILIDAD_DOCUMENTO', 'COHERENCIA_DOCUMENTO', 'CONTROL_CAMBIOS_DOC', 'VALIDACION_PROFESIONAL']


,Ruta,Archivo,Tipo,Responsables,Validacion_Profesional,Firmas_Aprobacion,Logo_Entidades,Validacion_Profesional_OK
0,pdfs/AVP-SAV-3000-D-EST-GEO-ING-000003.pdf,AVP-SAV-3000-D-EST-GEO-ING-000003.pdf,DOCUMENTO,SI,SI,SI,SI,SI
1,pdfs/AVP-SAV-9000-D-INF-CYT-ING-000001.pdf,AVP-SAV-9000-D-INF-CYT-ING-000001.pdf,DOCUMENTO,SI,SI,NO,SI,NO


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## (Opcional) Reentrenar los modelos
Requiere el **dataset anotado en CVAT** en `data/raw/` (no incluido en el repo por
confidencialidad/tamaño). Con tu dataset disponible, descomenta:

In [61]:
# !python src/convert.py --planos data/raw/planos --documentos data/raw/documentos --val-frac 0.15
# !python src/train.py --task both     # genera models/<task>/best.pt

In [90]:
!python scripts/diag_control.py --pdfs pdfs

In [91]:
from pathlib import Path
from IPython.display import Image, display

carpeta = Path("outputs/diag_crops")

for p in sorted(carpeta.glob("*.png")):
    if "fecha" in p.name.lower():
        print("\n" + p.name)
        display(Image(filename=str(p)))

In [92]:
from pathlib import Path
from PIL import Image
import sys

sys.path.insert(0, "src")
import extract as EX

for p in sorted(Path("outputs/diag_crops").glob("*fecha*.png")):
    print("\n", p.name)
    img = Image.open(p).convert("RGB")
    for psm in [6, 11, 4]:
        print("PSM", psm, "=>", EX.ocr_variants(img, psm=psm))